# データセットインポート

In [1]:
# torch
import torch
from torch import nn
from torch import optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

# numpy
import numpy as np

# other
import random
from tqdm import tqdm
from enum import Enum

# ランダムシード固定

In [2]:
is_fixing = True
if is_fixing:
    torch.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.default_rng(seed=42)
    random.seed(42)

# ハイパーパラメータ定義

In [ ]:
batch_size = 256
lr = 1e-4

# モデル定義

In [4]:
class MyResNet(nn.Module):

    def __init__(self, in_shape: tuple[int, int, int], out_size: int):
        '''
        param:
            in_shape: H, W, C
            out_size: output size of MyResNet
        '''
        super().__init__()

        


# データセットダウンロード + データローダー作成

## データセットはCIFAR-10 or CIFAR-100を使用．フラグにより切り替え

In [5]:
class CIFAR_Dataset(Enum):
    CIFAR10 = 0
    CIFAR100 = 1

In [6]:
used_dataset = CIFAR_Dataset.CIFAR10

if used_dataset == CIFAR_Dataset.CIFAR10:
    train_dataset = torchvision.datasets.CIFAR10(
        root="./dataset", 
        train=True, 
        transform=transforms.ToTensor(), 
        download=True
    )

    test_dataset = torchvision.datasets.CIFAR10(
        root="./dataset", 
        train=False, 
        transform=transforms.ToTensor(), 
        download=True
    )
elif used_dataset == CIFAR_Dataset == CIFAR_Dataset.CIFAR100:
    train_dataset = torchvision.datasets.CIFAR100(
        root="./dataset", 
        train=True, 
        transform=transforms.ToTensor(), 
        download=True
    )

    test_dataset = torchvision.datasets.CIFAR100(
        root="./dataset", 
        train=False, 
        transform=transforms.ToTensor(), 
        download=True
    )
else:
    used_dataset = CIFAR_Dataset.CIFAR10

    train_dataset = torchvision.datasets.CIFAR10(
        root="./dataset", 
        train=True, 
        transform=transforms.ToTensor(), 
        download=True
    )

    test_dataset = torchvision.datasets.CIFAR10(
        root="./dataset", 
        train=False, 
        transform=transforms.ToTensor(), 
        download=True
    )

100%|██████████| 170M/170M [00:43<00:00, 3.96MB/s] 
c:\work\study\DL_in_img\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


## DataLoader作成

In [7]:
train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset, 
    batch_size=batch_size, 
    shuffle=True
)

test_dataloader = torch.utils.data.DataLoader(
    dataset=test_dataset, 
    batch_size=256, 
    shuffle=True
)

# 学習 & 評価 関数定義

In [10]:
def train_one_epoch(
    model: nn.Module, 
    train_loader: torch.utils.data.DataLoader, 
    optimizer: torch.optim.Optimizer, 
    criterion: nn.Module, 
    device: torch.device
) -> tuple[float, float]:
    
    model.train()

    total_loss: float = 0.0
    num_samples: int = 0
    total_correct: int = 0

    for i, (bimgs, blabels) in enumerate(train_loader):
        bimgs: torch.Tensor = bimgs.to(device)
        blabels: torch.Tensor = blabels.to(device)

        # 推論
        outputs: torch.Tensor = model.forward(bimgs)
        preds: torch.Tensor = torch.argmax(outputs, dim=1)

        # 損失計算
        loss: torch.Tensor = criterion.forward(outputs, blabels)

        # パラメータ更新
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 集計
        num_samples = bimgs.shape[0]
        total_loss += loss.item() * bimgs.shape[0]
        total_correct += int((preds == blabels).sum().item())
    
    acc = total_correct / num_samples
    ave_loss = total_loss / num_samples

    return acc, ave_loss

def eval_model(
    model: nn.Module, 
    test_loader: torch.utils.data.DataLoader, 
    criterion: nn.Module, 
    device: torch.device
) -> tuple[float, float]:
    
    model.eval()

    num_sample: int = 0
    total_loss: float = 0.0
    total_correct: int = 0

    with torch.no_grad():
        for i, (bimgs, blabels) in enumerate(test_loader):
            bimgs: torch.Tensor = bimgs.to(device)
            blabels: torch.Tensor = blabels.to(device)

            # 推論
            outputs: torch.Tensor = model.forward(bimgs)
            preds: torch.Tensor = torch.argmax(outputs, dim=1)

            # 集計
            loss: torch.Tensor = criterion.forward(outputs, blabels)
            num_sample += bimgs.shape[0]
            total_loss += loss.item() * bimgs.shape[0]
            total_correct += int((preds == blabels).sum().item())
    
    acc = total_correct / num_sample
    ave_loss = total_loss / num_sample

    return acc, ave_loss

# 学習